# E-commerce Sales Analysis (Refactored v2)

Configurable analysis of delivered e-commerce sales: revenue, products, geography and customer experience for any month, year or date range, optionally compared with a second period.

## Table of Contents
1. [Introduction and Business Objectives](#1.-Introduction-and-Business-Objectives)
2. [Data Dictionary](#2.-Data-Dictionary)
3. [Data Loading and Configuration](#3.-Data-Loading-and-Configuration)
4. [Data Preparation and Transformation](#4.-Data-Preparation-and-Transformation)
5. [Business Metrics Calculation](#5.-Business-Metrics-Calculation)
    - 5.1 Revenue analysis
    - 5.2 Product analysis
    - 5.3 Geographic analysis
    - 5.4 Customer experience analysis
6. [Summary of Observations](#6.-Summary-of-Observations)

## 1. Introduction and Business Objectives

This notebook answers the following business questions for a chosen analysis period:

- How much revenue was generated, and how does it compare with the comparison period?
- What is the month-over-month revenue trend?
- What are the average order value and the number of orders?
- Which product categories generate the most revenue?
- Which states generate the most revenue?
- How satisfied are customers, and how does delivery time relate to review scores?
- What share of orders ends in each order status?

**How the code is organised**

| File | Responsibility |
|---|---|
| `data_loader_v2.py` | Loading, cleaning, joining and date filtering of the data |
| `business_metrics_v2.py` | Metric calculations only (no plotting or printing) |
| This notebook | Configuration, charts, presentation and observations |

All analysis periods are controlled from the configuration cell in section 3. Nothing else needs to be edited.

## 2. Data Dictionary

**Source tables** (CSV files in `ecommerce_data/`)

| Table | One row per | Used for |
|---|---|---|
| orders | Order | Status, purchase and delivery timestamps, customer link |
| order_items | Item within an order | Item price (revenue) and product link |
| products | Product | Product category |
| customers | Customer | Customer state |
| order_reviews | Review | Review score |

**Key columns**

| Column | Meaning |
|---|---|
| `order_id` | Unique identifier of an order. An order can contain several items. |
| `price` | Item price in currency units. Revenue is the sum of `price`; freight is excluded. |
| `order_status` | Lifecycle state of the order, for example delivered, shipped, canceled. |
| `order_purchase_timestamp` | Date and time the order was placed. Defines which period an order belongs to. |
| `order_delivered_customer_date` | Date and time the order reached the customer. |
| `purchase_year`, `purchase_month` | Year and month extracted from the purchase timestamp. |
| `delivery_days` | Whole days between purchase and delivery to the customer. |
| `product_category_name` | Category of the purchased product. |
| `customer_state` | State of the customer who placed the order. |
| `review_score` | Customer rating from 1 (lowest) to 5 (highest). Averaged per order if an order has several reviews. |

**Business terms**

| Term | Definition used in this notebook |
|---|---|
| Revenue | Sum of item `price` for orders in the selected status within the period |
| Average order value (AOV) | Mean of per-order revenue |
| Growth rate | (analysis period - comparison period) / comparison period |
| Mean monthly growth | Average of month-over-month revenue changes within the analysis period |
| Revenue share | Revenue of a group divided by total revenue of the period |

## 3. Data Loading and Configuration

Edit the values in the next cell to change the analysis. Use either a year (and optionally a month) or explicit start and end dates. When dates are given they take priority over the year and month.

In [1]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

import business_metrics_v2 as bm
import data_loader_v2 as dl

warnings.filterwarnings('ignore', category=FutureWarning)

# ----------------------------- Configuration -----------------------------
DATA_PATH = 'ecommerce_data/'

# Analysis period: a year, optionally narrowed to one month (1-12).
ANALYSIS_YEAR = 2023
ANALYSIS_MONTH = None            # None analyses the full year

# Comparison period. With ANALYSIS_MONTH set, the same month of this year is used.
COMPARISON_YEAR = 2022           # None disables all comparisons

# Optional explicit date ranges (inclusive, 'YYYY-MM-DD'). These override the year and month above.
START_DATE = None
END_DATE = None
COMPARISON_START_DATE = None
COMPARISON_END_DATE = None

# Order status to include in sales metrics. None includes every status.
ORDER_STATUS = 'delivered'

# Upper bounds (in days) of the delivery-time buckets; the last bucket is open-ended.
DELIVERY_BUCKET_EDGES = [3, 7]

# Number of categories shown in the category chart. None shows all categories.
TOP_N_CATEGORIES = None

In [2]:
tables = dl.load_raw_tables(DATA_PATH)
pd.DataFrame(
    {'rows': {name: len(df) for name, df in tables.items()},
     'columns': {name: df.shape[1] for name, df in tables.items()}}
)

,rows,columns
orders,10000,8
order_items,16047,7
products,6000,9
customers,8000,5
reviews,6571,7
payments,14091,5


## 4. Data Preparation and Transformation

The sales dataset has one row per order item and joins orders, products, customers and review scores. It is built once for all periods and then filtered to the analysis and comparison periods.

In [3]:
def select_period(year, month, start, end):
    """Return (sales, orders, label) for a period given as year/month or start/end dates."""
    if start is not None and end is not None:
        sales = dl.filter_by_period(all_sales, start, end)
        orders = dl.filter_by_period(all_orders, start, end)
        label = f'{start} to {end}'
    else:
        sales = dl.filter_by_year_month(all_sales, year, month)
        orders = dl.filter_by_year_month(all_orders, year, month)
        label = f'{year}' if month is None else f'{year}-{month:02d}'
    return sales, orders, label


all_sales = dl.build_sales_dataset(tables, status=ORDER_STATUS)
all_orders = dl.clean_orders(tables['orders'])

current, current_orders, current_label = select_period(
    ANALYSIS_YEAR, ANALYSIS_MONTH, START_DATE, END_DATE)
if current.empty:
    raise ValueError(f'No sales found for the analysis period {current_label}.')

has_comparison = COMPARISON_YEAR is not None or (
    COMPARISON_START_DATE is not None and COMPARISON_END_DATE is not None)
if has_comparison:
    previous, previous_orders, previous_label = select_period(
        COMPARISON_YEAR, ANALYSIS_MONTH, COMPARISON_START_DATE, COMPARISON_END_DATE)
    if previous.empty:
        print(f'No sales found for the comparison period {previous_label}; comparisons are skipped.')
        has_comparison = False

status_text = ORDER_STATUS if ORDER_STATUS else 'all statuses'
print(f'Analysis period  : {current_label} ({len(current):,} order items, {bm.order_count(current):,} orders)')
if has_comparison:
    print(f'Comparison period: {previous_label} ({len(previous):,} order items, {bm.order_count(previous):,} orders)')
print(f'Order status     : {status_text}')

Analysis period  : 2023 (7,448 order items, 4,635 orders)
Comparison period: 2022 (7,641 order items, 4,749 orders)
Order status     : delivered


In [4]:
current.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,customer_id,order_status,order_purchase_timestamp,order_delivered_customer_date,purchase_year,purchase_month,delivery_days,product_category_name,customer_state,review_score
0,ord_5fa044951857e02fd1347b47,1,prod_9b6f1793388ae983f852d189,sell_34212979d97b6d1a8409a01e,2025-07-19 21:15:46.994884,111.91,9.50,cust_2d36e4b647be13c15df550af,delivered,2023-04-24 13:05:32.579655,2023-05-01 02:05:32.579655,2023,4,6.0,grocery_gourmet_food,TN,5.0
1,ord_5fa044951857e02fd1347b47,2,prod_c7b5997a4d0061a9f50e874c,sell_4668fb88b81f8d834371c335,2025-07-17 10:00:54.671461,878.42,77.50,cust_2d36e4b647be13c15df550af,delivered,2023-04-24 13:05:32.579655,2023-05-01 02:05:32.579655,2023,4,6.0,electronics,TN,5.0
5,ord_43b53981d951f855231d09ec,1,prod_496e129d773e9587e8ddc065,sell_6cfe47a34211c0b92f0fd126,2025-07-13 21:00:53.795592,749.83,13.94,cust_e670ce607e4f6aa14935ece0,delivered,2023-12-10 05:13:58.249202,2023-12-19 09:13:58.249202,2023,12,9.0,sports_outdoors,FL,5.0
6,ord_e60b1e267fd32d93c4d0745b,1,prod_139fc7c40b163ec81342741e,sell_bcb2c47f4631ff00586f7de0,2025-07-14 01:28:11.006814,361.54,54.65,cust_deec5ec61bd39869531896b3,delivered,2023-04-11 05:42:57.214593,2023-04-22 20:42:57.214593,2023,4,11.0,home_garden,PA,5.0
7,ord_e60b1e267fd32d93c4d0745b,2,prod_8668b3ebccd7183bae97138a,sell_bc4ff6f9c9f971b8fc9bf8a3,2025-07-26 07:31:46.754270,25.59,2.57,cust_deec5ec61bd39869531896b3,delivered,2023-04-11 05:42:57.214593,2023-04-22 20:42:57.214593,2023,4,11.0,grocery_gourmet_food,PA,5.0


### Chart settings and helper functions

Line charts use deep teal for the analysis period and grey for the comparison period. Bar charts give every bar its own colour from a qualitative palette, and the map uses a single teal scale. Colours carry no meaning such as good or bad. Every title carries the period it covers.

In [5]:
COLOR_PRIMARY = '#0F766E'
COLOR_COMPARISON = '#94A3B8'
COLOR_SCALE = 'Teal'
# Qualitative palette for bar charts: one distinct colour per bar, none shared with the v1 notebook defaults.
BAR_PALETTE = ['#0F766E', '#B45309', '#4338CA', '#BE123C', '#4D7C0F', '#7E22CE', '#0E7490',
               '#A16207', '#9D174D', '#475569', '#15803D', '#C2410C', '#1D4ED8']
MONTH_NAMES = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
LAYOUT = dict(template='plotly_white', font=dict(size=13), margin=dict(t=80, b=60))


def bar_colors(n):
    """Return the first n colours of BAR_PALETTE, one per bar."""
    return BAR_PALETTE[:n]


def fmt_money(value):
    """Format a currency amount with thousands separators, e.g. 3,360,294.74."""
    return 'n/a' if pd.isna(value) else f'{value:,.2f}'


def fmt_pct(value, decimals=2):
    """Format a fraction as a signed percentage, e.g. -2.46%."""
    return 'n/a' if pd.isna(value) else f'{value * 100:+.{decimals}f}%'


def plot_monthly_revenue(current_series, current_label, previous_series=None, previous_label=None):
    """Line chart of revenue per month; the comparison period is aligned by calendar month."""
    fig = go.Figure()
    multi_year = current_series.index.min().year != current_series.index.max().year

    if previous_series is not None and not multi_year:
        fig.add_trace(go.Scatter(
            x=[MONTH_NAMES[p.month - 1] for p in previous_series.index], y=previous_series.values,
            mode='lines+markers', name=previous_label,
            line=dict(color=COLOR_COMPARISON, dash='dash')))
    x_current = (current_series.index.strftime('%Y-%m') if multi_year
                 else [MONTH_NAMES[p.month - 1] for p in current_series.index])
    fig.add_trace(go.Scatter(
        x=list(x_current), y=current_series.values, mode='lines+markers',
        name=current_label, line=dict(color=COLOR_PRIMARY, width=3)))

    fig.update_layout(title=f'Monthly Revenue, {current_label}',
                      xaxis_title='Purchase month', yaxis_title='Revenue (currency units)',
                      yaxis_tickformat=',', legend_title_text='Period', **LAYOUT)
    return fig


def plot_category_revenue(category_df, label):
    """Bar chart of revenue per product category, largest first."""
    fig = px.bar(category_df, x='product_category_name', y='revenue',
                 color='product_category_name', color_discrete_sequence=BAR_PALETTE)
    fig.update_layout(title=f'Revenue by Product Category, {label}', showlegend=False,
                      xaxis_title='Product category', yaxis_title='Revenue (currency units)',
                      yaxis_tickformat=',', xaxis_tickangle=-45, **LAYOUT)
    return fig


def plot_state_map(state_df, label):
    """US choropleth of revenue per customer state."""
    fig = px.choropleth(state_df, locations='customer_state', color='revenue',
                        locationmode='USA-states', scope='usa',
                        color_continuous_scale=COLOR_SCALE,
                        labels={'revenue': 'Revenue (currency units)', 'customer_state': 'State'})
    fig.update_layout(title=f'Revenue by Customer State, {label}', **LAYOUT)
    return fig


def plot_review_distribution(distribution, label):
    """Horizontal bar chart of the share of reviewed orders per review score."""
    fig = go.Figure(go.Bar(
        x=distribution.values * 100, y=[str(int(s)) for s in distribution.index],
        orientation='h', marker_color=bar_colors(len(distribution)),
        text=[f'{v * 100:.1f}%' for v in distribution.values], textposition='outside'))
    fig.update_layout(title=f'Review Score Distribution, {label}',
                      xaxis_title='Share of reviewed orders (%)', yaxis_title='Review score (1 = lowest, 5 = highest)',
                      yaxis=dict(categoryorder='category descending'), **LAYOUT)
    return fig


def plot_review_by_delivery_days(df, label):
    """Line chart of mean review score for each whole number of delivery days."""
    fig = go.Figure(go.Scatter(x=df['delivery_days'], y=df['review_score'],
                               mode='lines+markers', line=dict(color=COLOR_PRIMARY),
                               customdata=df['orders'],
                               hovertemplate='Delivery days: %{x}<br>Mean review score: %{y:.2f}<br>Orders: %{customdata}<extra></extra>'))
    fig.update_layout(title=f'Mean Review Score by Delivery Time, {label}',
                      xaxis_title='Delivery time (days from purchase)', yaxis_title='Mean review score (1-5)',
                      yaxis_range=[1, 5], **LAYOUT)
    return fig


def plot_review_by_bucket(df, label):
    """Bar chart of mean review score per delivery-time bucket."""
    fig = go.Figure(go.Bar(x=df['delivery_bucket'].astype(str), y=df['review_score'],
                           marker_color=bar_colors(len(df)),
                           text=[f'{v:.2f}' for v in df['review_score']], textposition='outside'))
    fig.update_layout(title=f'Mean Review Score by Delivery Time Bucket, {label}',
                      xaxis_title='Delivery time (days from purchase)', yaxis_title='Mean review score (1-5)',
                      yaxis_range=[1, 5], **LAYOUT)
    return fig


def plot_status_mix(status_share, label):
    """Bar chart of the share of orders per order status."""
    fig = go.Figure(go.Bar(x=status_share.index, y=status_share.values * 100,
                           marker_color=bar_colors(len(status_share)),
                           text=[f'{v * 100:.1f}%' for v in status_share.values], textposition='outside'))
    fig.update_layout(title=f'Order Status Mix, {label}',
                      xaxis_title='Order status', yaxis_title='Share of orders (%)', **LAYOUT)
    return fig

## 5. Business Metrics Calculation

### 5.1 Revenue Analysis

Total revenue, average order value and order count for the analysis period, with the change against the comparison period when one is configured.

In [6]:
def revenue_table():
    """Headline revenue metrics for the analysis period and, if configured, the comparison period."""
    rows = {
        'Total revenue': (bm.total_revenue(current), bm.total_revenue(previous) if has_comparison else None),
        'Average order value': (bm.average_order_value(current), bm.average_order_value(previous) if has_comparison else None),
        'Number of orders': (bm.order_count(current), bm.order_count(previous) if has_comparison else None),
    }
    table = pd.DataFrame(rows, index=[current_label, 'comparison']).T
    if has_comparison:
        table.columns = [current_label, previous_label]
        table['change'] = [bm.growth_rate(cur, prev) for cur, prev in zip(table.iloc[:, 0], table.iloc[:, 1])]
    else:
        table = table.iloc[:, [0]]
        table.columns = [current_label]
    return table


metrics_table = revenue_table()
display_table = metrics_table.copy()
for col in display_table.columns:
    if col == 'change':
        display_table[col] = display_table[col].map(fmt_pct)
    else:
        display_table[col] = [f'{v:,.0f}' if name == 'Number of orders' else fmt_money(v)
                              for name, v in display_table[col].items()]
display_table

,2023,2022,change
Total revenue,"3,360,294.74","3,445,076.96",-2.46%
Average order value,724.98,725.43,-0.06%
Number of orders,"4,635","4,749",-2.40%


#### Monthly revenue trend

Revenue per purchase month. The mean monthly growth is the average of the month-over-month changes within the analysis period.

In [7]:
monthly_current = bm.monthly_revenue(current)
monthly_previous = bm.monthly_revenue(previous) if has_comparison else None
mean_growth = bm.mean_monthly_growth(current)

print(f'Mean month-over-month revenue growth, {current_label}: {fmt_pct(mean_growth)}')
plot_monthly_revenue(monthly_current, current_label, monthly_previous,
                     previous_label if has_comparison else None).show()
monthly_current.rename('revenue').to_frame().assign(
    mom_growth=lambda d: d['revenue'].pct_change().map(fmt_pct)
).rename_axis('month')

Mean month-over-month revenue growth, 2023: -0.39%


,revenue,mom_growth
month,,
2023-01,284855.48,n/a
2023-02,276758.54,-2.84%
2023-03,285618.77,+3.20%
2023-04,253794.62,-11.14%
2023-05,301347.96,+18.74%
2023-06,289126.72,-4.06%
2023-07,296209.39,+2.45%
2023-08,286645.60,-3.23%
2023-09,303793.15,+5.98%


### 5.2 Product Analysis

Revenue per product category, largest first, with each category's share of total revenue.

In [8]:
category_revenue = bm.revenue_by_category(current, top_n=TOP_N_CATEGORIES)
plot_category_revenue(category_revenue, current_label).show()
category_revenue.assign(
    revenue=category_revenue['revenue'].map(fmt_money),
    revenue_share=category_revenue['revenue_share'].map(lambda v: f'{v * 100:.1f}%'),
)

,product_category_name,revenue,revenue_share
0,electronics,"1,401,359.21",41.7%
1,home_garden,"862,653.27",25.7%
2,sports_outdoors,"278,845.32",8.3%
3,automotive,"247,707.10",7.4%
4,clothing_shoes_jewelry,"232,745.04",6.9%
5,toys_games,"70,163.64",2.1%
6,health_personal_care,"65,370.27",1.9%
7,tools_home_improvement,"54,280.33",1.6%
8,beauty_personal_care,"49,213.30",1.5%
9,books_media,"38,559.08",1.1%


### 5.3 Geographic Analysis

Revenue by the state of the purchasing customer.

In [9]:
state_revenue = bm.revenue_by_state(current)
plot_state_map(state_revenue, current_label).show()
state_revenue.head(10).assign(revenue=lambda d: d['revenue'].map(fmt_money))

,customer_state,revenue
0,CA,"537,880.74"
1,TX,"403,303.49"
2,FL,"310,195.84"
3,NY,"253,111.85"
4,IL,"185,930.89"
5,PA,"173,260.68"
6,OH,"157,619.72"
7,GA,"146,414.65"
8,MI,"132,325.60"
9,NC,"128,314.74"


### 5.4 Customer Experience Analysis

Review scores, delivery time and order status. Review and delivery metrics are calculated once per order, so orders with several items are not counted more than once.

In [10]:
avg_review = bm.average_review_score(current)
avg_delivery = bm.average_delivery_days(current)
print(f'Average review score, {current_label}: {avg_review:.2f} (scale 1-5)')
print(f'Average delivery time, {current_label}: {avg_delivery:.2f} days')
if has_comparison:
    print(f'Average review score, {previous_label}: {bm.average_review_score(previous):.2f}')
    print(f'Average delivery time, {previous_label}: {bm.average_delivery_days(previous):.2f} days')

plot_review_distribution(bm.review_score_distribution(current), current_label).show()

Average review score, 2023: 4.10 (scale 1-5)
Average delivery time, 2023: 8.03 days
Average review score, 2022: 4.11
Average delivery time, 2022: 8.06 days


#### Review score and delivery time

In [11]:
review_days = bm.review_by_delivery_days(current)
plot_review_by_delivery_days(review_days, current_label).show()
review_days

,delivery_days,review_score,orders
0,2.0,4.033898,59
1,3.0,4.248555,173
2,4.0,4.068000,250
3,5.0,4.054054,296
4,6.0,4.073529,340
5,7.0,4.102236,313
6,8.0,3.965418,347
7,9.0,4.117470,332
8,10.0,4.100917,327
9,11.0,4.146580,307


In [12]:
review_bucket = bm.review_by_delivery_bucket(current, DELIVERY_BUCKET_EDGES)
plot_review_by_bucket(review_bucket, current_label).show()
review_bucket

,delivery_bucket,review_score,orders
0,1-3 days,4.193966,232
1,4-7 days,4.075063,1199
2,8+ days,4.108138,1794


#### Order status mix

Share of all orders placed in the period by status. This uses orders of every status, not only the status selected for sales metrics.

In [13]:
status_mix = bm.order_status_mix(current_orders)
plot_status_mix(status_mix, current_label).show()
status_mix.map(lambda v: f'{v * 100:.2f}%').to_frame()

,share
order_status,
delivered,93.64%
shipped,3.19%
canceled,1.29%
processing,1.19%
pending,0.51%
returned,0.18%


## 6. Summary of Observations

The text below is generated from the configured analysis and updates automatically when the configuration changes.

In [14]:
def describe_change(name, change, comparison_label):
    """Return a sentence describing a change against the comparison period."""
    if pd.isna(change):
        return ''
    direction = 'higher' if change > 0 else 'lower' if change < 0 else 'unchanged'
    return f' ({name} is {abs(change) * 100:.2f}% {direction} than {comparison_label})' if direction != 'unchanged' else f' ({name} is unchanged versus {comparison_label})'


cmp_label = previous_label if has_comparison else None
change = metrics_table['change'] if has_comparison else {}
top_category = category_revenue.iloc[0]
top_state = state_revenue.iloc[0]
best_month = monthly_current.idxmax()
worst_month = monthly_current.idxmin()

lines = [
    f'Period analysed: {current_label}; order status: {status_text}.',
    f'Revenue: {fmt_money(bm.total_revenue(current))}'
    + (describe_change('revenue', change['Total revenue'], cmp_label) if has_comparison else '') + '.',
    f'Orders: {bm.order_count(current):,}'
    + (describe_change('order count', change['Number of orders'], cmp_label) if has_comparison else '')
    + f'; average order value {fmt_money(bm.average_order_value(current))}'
    + (describe_change('AOV', change['Average order value'], cmp_label) if has_comparison else '') + '.',
    f'Trend: mean month-over-month revenue growth is {fmt_pct(mean_growth)}; highest month {best_month} '
    f'({fmt_money(monthly_current.max())}), lowest month {worst_month} ({fmt_money(monthly_current.min())}).',
    f'Products: {top_category["product_category_name"]} is the largest category with '
    f'{top_category["revenue_share"] * 100:.1f}% of revenue ({len(category_revenue)} categories shown).',
    f'Geography: {top_state["customer_state"]} is the largest state with '
    f'{top_state["revenue"] / bm.total_revenue(current) * 100:.1f}% of revenue ({len(state_revenue)} states with sales).',
    f'Customer experience: average review score {avg_review:.2f} out of 5; average delivery time {avg_delivery:.1f} days.',
    'Delivery buckets: ' + '; '.join(
        f'{row.delivery_bucket} has mean review {row.review_score:.2f} ({row.orders:,} orders)'
        for row in review_bucket.itertuples()) + '.',
    'Order status: ' + ', '.join(f'{s} {v * 100:.1f}%' for s, v in status_mix.items()) + '.',
]
print('\n\n'.join(lines))

Period analysed: 2023; order status: delivered.

Revenue: 3,360,294.74 (revenue is 2.46% lower than 2022).

Orders: 4,635 (order count is 2.40% lower than 2022); average order value 724.98 (AOV is 0.06% lower than 2022).

Trend: mean month-over-month revenue growth is -0.39%; highest month 2023-09 (303,793.15), lowest month 2023-04 (253,794.62).

Products: electronics is the largest category with 41.7% of revenue (13 categories shown).

Geography: CA is the largest state with 16.0% of revenue (20 states with sales).

Customer experience: average review score 4.10 out of 5; average delivery time 8.0 days.

Delivery buckets: 1-3 days has mean review 4.19 (232 orders); 4-7 days has mean review 4.08 (1,199 orders); 8+ days has mean review 4.11 (1,794 orders).

Order status: delivered 93.6%, shipped 3.2%, canceled 1.3%, processing 1.2%, pending 0.5%, returned 0.2%.
